In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

# Load the persisted data from Notebook 1
df_clean = pd.read_parquet('group2_stocks.parquet')
df_benchmark = pd.read_parquet('benchmark.parquet')

print(f"✅ Data loaded successfully. stocks: {df_clean.shape}, benchmark: {df_benchmark.shape}")

### Core Analytics Engineering

In [ ]:
# --- CELL 1: PRICE PROFILE CALCULATIONS ---

# 1. Calculate Daily Log Returns
# Mathematically: $r_t = \ln(P_t / P_{t-1})$
# This provides stationarity, which is essential for stable clustering.
log_returns = np.log(df_clean / df_clean.shift(1)).fillna(0)

# 2. Calculate Cumulative Log Returns (For Visualization/Growth Profile)
# This shows the total "path" of each stock over the 2-year window.
cum_log_returns = log_returns.cumsum()

# 3. Benchmark Log Returns (S&P 500)
log_ret_bench = np.log(df_benchmark / df_benchmark.shift(1)).fillna(0)
cum_ret_bench = log_ret_bench.cumsum()

# 4. Prepare the Feature Matrix for Clustering
# In Price Profile clustering, each stock is a row, and each trading day's return is a feature.
# We transpose the dataframe so the shape is (Number of Stocks, Number of Days)
price_feature_matrix = log_returns.T 

print("✅ Price Profiles Created.")
print(f"📊 Feature Matrix Shape (Stocks x Days): {price_feature_matrix.shape}")
print(f"📈 Date Range: {log_returns.index[0].date()} to {log_returns.index[-1].date()}")

### Visual EDA: Price Action & Growth Comparison:

In [ ]:
# --- CELL 2: PLOT PRICE & RETURNS ---
# Plot A: Raw Prices (Linear Scale)
fig_raw = go.Figure()
for ticker in df_clean.columns:
    fig_raw.add_trace(go.Scatter(
        x=df_clean.index, y=df_clean[ticker], name=ticker,
        mode='lines', line=dict(width=1),
        hovertemplate=f"<b>{ticker}</b><br>Price: $%{{y:.2f}}<extra></extra>"
    ))

fig_raw.update_layout(
    title="Price History (Linear Scale)", 
    # yaxis_type="log",  <-- REMOVED to revert to normal scale
    yaxis_title="Price ($)", 
    template="plotly_white", 
    height=500
)
fig_raw.show()

# Plot B: Cumulative Returns
fig_ret = go.Figure()
for ticker in cum_log_returns.columns:
    fig_ret.add_trace(go.Scatter(
        x=cum_log_returns.index, y=cum_log_returns[ticker], name=ticker,
        mode='lines', line=dict(width=1),
        hovertemplate=f"<b>{ticker}</b><br>Return: %{{y:.2%}}<extra></extra>"
    ))

fig_ret.update_layout(
    title="Cumulative Log Returns", 
    yaxis_title="Cumulative Return", 
    template="plotly_white", 
    height=500
)
fig_ret.show()

### Structural Stratification

In [ ]:
# --- CELL 3: STRUCTURAL CLASSIFICATION ---
# Calculate summary stats from the already downloaded data
stats = pd.DataFrame({'Max': df_clean.max(), 'Min': df_clean.min(), 'Mean': df_clean.mean()})

# Define Bins
bins = [0, 100, 200, 400, 700, float('inf')]
labels = ["1. Micro ($0-100)", "2. Small ($100-200)", "3. Medium ($200-400)", "4. Large ($400-700)", "5. Mega (>$700)"]
stats['Class'] = pd.cut(stats['Max'], bins=bins, labels=labels)

# Plot each class
for label in labels:
    subset = stats[stats['Class'] == label].sort_values(by='Max')
    if subset.empty: continue

    fig = go.Figure()
    # Add Range Lines
    for ticker in subset.index:
        fig.add_shape(type="line", x0=ticker, x1=ticker, 
                      y0=subset.loc[ticker, 'Min'], y1=subset.loc[ticker, 'Max'],
                      line=dict(color="gray", width=1))
    
    # Add Markers
    fig.add_trace(go.Scatter(x=subset.index, y=subset['Max'], mode='markers', name='Max', marker=dict(color='red', size=8)))
    fig.add_trace(go.Scatter(x=subset.index, y=subset['Min'], mode='markers', name='Min', marker=dict(color='green', size=8)))
    fig.add_trace(go.Scatter(x=subset.index, y=subset['Mean'], mode='markers', name='Mean', marker=dict(color='blue', size=5, symbol='diamond')))
    
    fig.update_layout(title=f"Structure: {label}", xaxis_title="Ticker", yaxis_title="Price ($)", template="plotly_white", height=300)
    fig.show()

### Benchmark Overlay

In [ ]:
# --- CELL 6: STATIC BENCHMARK COMPARISON (MATPLOTLIB) ---
# This cell uses the pre-calculated data from Cells 2 & 3 to avoid repetition.

# --- METHOD A: RAW PRICE "SPAGHETTI" (SINGLE Y-AXIS) ---
plt.figure(figsize=(14, 7))

# Plot Stocks (Using df_clean from Cell 2)
plt.plot(df_clean, lw=0.5, alpha=0.4)

# Plot S&P 500 on the same axis (Using df_benchmark from Cell 2)
plt.plot(df_benchmark, color='black', lw=3, label='S&P 500 Benchmark (^GSPC)')

plt.title("Method A: Raw Adjusted Prices (Single Axis with Benchmark)")
plt.xlabel("Date")
plt.ylabel("Price ($)")
plt.grid(True, alpha=0.2)
plt.legend(loc="upper left") # Matplotlib will only show the labeled Benchmark line
plt.show()

# --- METHOD B: CUMULATIVE LOG RETURNS ---
plt.figure(figsize=(14, 7))

# Plot Stock Returns (Using cum_log_returns from Cell 3)
plt.plot(cum_log_returns, lw=0.8, alpha=0.3)

# Plot Benchmark Returns (Using cum_ret_bench from Cell 3)
# Note: We rely on cum_ret_bench calculated in Cell 3
plt.plot(cum_ret_bench, color='red', lw=3, label='S&P 500 Benchmark')

plt.axhline(0, color='black', lw=1, ls='--')
plt.title("Method B: Cumulative Log Returns vs. S&P 500 Benchmark")
plt.xlabel("Date")
plt.ylabel("Cumulative Log Return")
plt.legend(loc="upper left") # Will only show the Benchmark label
plt.grid(True, alpha=0.3)
plt.show()